# MeChess 2/3: train the chess-text language model (GPU notebook)

Only an adapter: it runs `chessme books-preflight` and `chessme books-nlp-train`. The model reads a comment or paragraph and predicts **which chess concepts it discusses** (keywords hidden, so it must use context),
**the human verdict on the move** (`?? ? ?! !? ! !!`) and **who stands better**. See `plan.md` section 8p and `chessme/books/nlp.py`.

**Before you spend GPU hours**
1. Run notebook 1 first and add its output as an input (*Add Input -> Notebook output files*). To continue an earlier training run, also add *this* notebook's earlier output as an input: the checkpoint is restored automatically.
2. *Accelerator -> GPU*, *Internet -> On*, set `REPO_URL`, **Save Version -> Save & Run All (Commit)**.
3. Steps 1 and 2 protect the quota: preflight (GPU, dependencies, the model downloads), then a **dry-run** of the whole path that also checks the model can *learn* (it must memorise 64 examples). If either fails, the notebook stops within minutes.
4. The real run has a **time budget** (`BUDGET_MIN`, default 600 min, below Kaggle's 12 h): at the deadline it saves a checkpoint and ends normally, so the output is kept. It logs validation numbers each epoch
   (concept average precision vs a random ranking, judgement / evaluation accuracy vs the majority guess) and warns if concepts are not clearly above random. **Suggested first session: `EPOCHS = 1`.**

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
MODEL = "distilroberta-base"                            # any Hugging Face encoder; a fast trial: "google/bert_uncased_L-2_H-128_A-2" (avoid DeBERTa unless sentencepiece is installed)
EPOCHS = 3
BATCH = 64
BUDGET_MIN = 600                                        # wall-clock budget of the real run, minutes
OUT = "/kaggle/working/learn" if os.path.exists("/kaggle") else "learn_local"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):                      # Kaggle: fetch the repository (Internet must be on)
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

## Step 1. Preflight: a GPU is present, dependencies import, the model can be downloaded

In [ ]:
sh(*CLI, "books-preflight", "--out", OUT, "--need-gpu", "--backend", "transformer", "--model", MODEL, "--no-network")

## Step 2. Dry-run: the full path on a few examples, and a check that the loss goes down (about a minute)

In [ ]:
sh(*CLI, "books-nlp-train", "--data", OUT, "--input-root", "/kaggle/input", "--out", f"{OUT}/nlp_dry", "--backend", "transformer", "--model", MODEL, "--dry-run", "--log", f"{OUT}/nlp_dry.log")

## Step 3. The real run (checkpoints regularly; stops cleanly at the time budget)

In [ ]:
sh(*CLI, "books-nlp-train", "--data", OUT, "--input-root", "/kaggle/input", "--out", f"{OUT}/nlp", "--backend", "transformer", "--model", MODEL,
   "--epochs", str(EPOCHS), "--batch", str(BATCH), "--deadline-minutes", str(BUDGET_MIN), "--ckpt-every", "500", "--slim", "--log", f"{OUT}/nlp.log")

## Step 4. Results and a few predictions

In [ ]:
sh(*CLI, "books-nlp-report", f"{OUT}/nlp")
sh(*CLI, "books-nlp-predict", f"{OUT}/nlp",
   "The knight can never be dislodged from d5 because no enemy pawn can attack that square.",
   "A terrible move, it loses a piece for nothing.",
   "Black is now completely lost.")